# 2.6 Warnings: mould detection from photos (first test on Kaggle)

**What this notebook does.** It trains a small model that draws a box around mould in a photo and says whether it is **Trichoderma** (green mould) or **mould** (everything else: Mold, Cobweb, Neurospora). It is the contamination part of the **Warnings** model for the cultivation planner.

**Data.** Three public Roboflow sets by the owner *baibua* (CC BY 4.0: credit the owner whenever results are shown). The newest version of each is used.

| Set | Roboflow project | Note |
|---|---|---|
| wwzez | baibua / mold-detection-training-wwzez | Trichoderma boxes look right |
| f3os8 | baibua / mold-detect-s-f3os8 | small set |
| 4n7zr | baibua / molds-4n7zr | real mould labels on many backgrounds (mycelium on many substrates and containers). This variety is wanted: user photos will look like that |

Plus **clean dish photos** (no mould, no boxes) from your own private Kaggle dataset `d005-clean-dishes`: the 8 substrate folders (FD, FLT, FLU, FSW, Flax, H, S, W) of d005 (Elsacker 2019), whole dishes photographed from above. They teach the model that a healthy white dish is not mould. d003 is left out: its images are thin curve strips (about 78×435 px), not whole dishes.

**Honest counting.** Roboflow adds flipped or rotated copies of one photo; we keep one copy per source photo. The same photo can also sit in two sets, or appear resized; a perceptual hash finds those and we keep one. The split into train and held-out is by photo, after duplicates are removed. Clean photos are split by dish, so photos of the same dish never sit on both sides.

**Never used here:** syldd photos. They are a final check only, and the notebook stops if a clean-photo path contains "syldd".

**Limit to remember.** The held-out 20% also picks the best epoch, so scores are slightly optimistic. The leave-one-source-out cell at the end (once a second licensed mould set exists) is the stronger test.

**Before you press Run All (once):**
1. Right panel, *Session options*: Accelerator = **GPU T4 x2** (or P100), Internet = **On**.
2. *Add-ons, Secrets*: switch `ROBOFLOW_API_KEY` on for this notebook. The key is never printed or saved.
3. *Add Input*: add your clean-dish dataset, then check `CLEAN_DIR` in the clearly marked cell (step 5): copy the real path with the copy icon in the Input panel.

At the end a small table with precision, recall, mAP50 and the false-alarm counts sums up everything; a screenshot of it is enough. The best weights are saved as `mould_best.pt` in the Output tab.

In [ ]:
# 1. Install the libraries we need (Kaggle has the rest)
!pip install -q roboflow ultralytics imagehash

In [ ]:
# 2. Settings: change these lines only if you want to
import json, random, re, shutil
from collections import Counter
from pathlib import Path

SETS = {
    # short name: (Roboflow workspace, project)
    "wwzez": ("baibua", "mold-detection-training-wwzez"),
    "f3os8": ("baibua", "mold-detect-s-f3os8"),
    "4n7zr": ("baibua", "molds-4n7zr"),
}
# their class name (lower case) -> our class number
CLASS_MAP = {"trichoderma": 0, "mold": 1, "mould": 1, "cobweb": 1, "neurospora": 1}
OUR_NAMES = {0: "Trichoderma", 1: "mould"}
HASH_DISTANCE = 4        # photos whose perceptual hashes differ in <= this many of 64 bits count as the same photo
HELD_OUT = 0.20          # share of photos (and of clean dishes) kept for testing
EPOCHS = 50
IMG_SIZE = 640
MODEL = "yolo11n.pt"     # smallest box model: fast, good enough for a first test
SEED = 0
WORK = Path("/kaggle/working")
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
random.seed(SEED)

## Step 1-2. Download the three sets
The key is read from the Kaggle secret and handed straight to Roboflow. It is not printed or written to any file.

In [ ]:
# 3. Download the newest version of each set in YOLO box format
import yaml
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow
rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))

downloads = {}   # short name -> (folder, {class number in that set: class name})
for short, (ws, proj_name) in SETS.items():
    proj = rf.workspace(ws).project(proj_name)
    ver = max(int(str(v.version).split("/")[-1]) for v in proj.versions())
    loc = proj.version(ver).download("yolov11", location=str(WORK / "raw" / short), overwrite=True).location
    names = yaml.safe_load(open(Path(loc) / "data.yaml"))["names"]
    names = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
    downloads[short] = (Path(loc), names)
    print(f"{short}: {ws}/{proj_name} version {ver}, their classes: {names}")

In [ ]:
# 4. List every photo with its boxes (their class names for now)
def read_boxes(label_file, names):
    boxes = []
    if not label_file.exists():
        return boxes
    for line in label_file.read_text().split("\n"):
        p = line.split()
        if len(p) < 5:
            continue
        cls = names.get(int(float(p[0])), "?")
        if len(p) == 5:
            x, y, w, h = map(float, p[1:])
        else:   # outline (polygon) labels: use the box around the outline
            xs, ys = [float(v) for v in p[1::2]], [float(v) for v in p[2::2]]
            x, y, w, h = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)
        boxes.append((cls, x, y, w, h))
    return boxes

raw = []
for short, (root, names) in downloads.items():
    for img in sorted(root.rglob("*")):
        if img.suffix.lower() in IMG_EXT and img.parent.name == "images":
            lab = img.parent.parent / "labels" / (img.stem + ".txt")
            raw.append({"path": img, "src": short, "boxes": read_boxes(lab, names)})
print("photo files found:", Counter(r["src"] for r in raw))

## Eye check of the labels
Twelve random photos per set with their boxes (their own class names). Look before trusting any score: boxes should sit on mould, not on healthy mycelium.

In [ ]:
# 5. Show 12 random boxed photos per set
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

def draw(rec):
    im = Image.open(rec["path"]).convert("RGB")
    d = ImageDraw.Draw(im)
    W, H = im.size
    for cls, x, y, w, h in rec["boxes"]:
        d.rectangle([(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H], outline=(255, 0, 0), width=max(2, W // 200))
    return im

for short in SETS:
    pool = [r for r in raw if r["src"] == short and r["boxes"]]
    pick = random.sample(pool, min(12, len(pool)))
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    for ax in axes.flat:
        ax.axis("off")
    for ax, r in zip(axes.flat, pick):
        ax.imshow(draw(r))
        ax.set_title(", ".join(sorted({b[0] for b in r["boxes"]})), fontsize=9)
    fig.suptitle(f"{short}: {len(pool)} photos with boxes", fontsize=14)
    plt.tight_layout()
    plt.show()

## Step 3. Remove copies and duplicates
First one copy per source photo (Roboflow names copies `<name>.rf.<code>.jpg`), then a perceptual hash across all three sets so resized or re-uploaded copies count too.

In [ ]:
# 6. One copy per source photo, then perceptual-hash duplicates across the sets
import numpy as np, imagehash

# a) Roboflow augmented copies: same text before ".rf."
seen, no_aug = set(), []
for r in raw:
    key = (r["src"], r["path"].name.split(".rf.")[0])
    if key not in seen:
        seen.add(key)
        no_aug.append(r)
print(f"files: {len(raw)} -> source photos after dropping Roboflow copies: {len(no_aug)}")

# b) perceptual hash
def phash_bits(path):
    with Image.open(path) as im:
        return imagehash.phash(im.convert("RGB")).hash.flatten().astype(np.int16)

recs, bits = [], []
for r in no_aug:
    try:
        bits.append(phash_bits(r["path"]))
        recs.append(r)
    except Exception as e:
        print("skipped unreadable photo:", r["path"].name, e)
B = np.stack(bits)
dist = 64 - (B @ B.T + (1 - B) @ (1 - B).T)

parent = list(range(len(recs)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
for i, j in np.argwhere(np.triu(dist <= HASH_DISTANCE, 1)):
    parent[find(i)] = find(j)

clusters = {}
for i in range(len(recs)):
    clusters.setdefault(find(i), []).append(i)
unique, cross_set = [], 0
for members in clusters.values():
    best = max(members, key=lambda i: (len(recs[i]["boxes"]), -i))   # keep the copy with most boxes
    unique.append(recs[best])
    cross_set += len({recs[i]["src"] for i in members}) > 1
print(f"duplicate groups spanning more than one set: {cross_set}")
print(f"UNIQUE PHOTOS: {len(unique)}  (from {len(raw)} files)")
print("unique photos per set:", dict(Counter(r["src"] for r in unique)))

## Step 4. Map the classes
Trichoderma stays **Trichoderma** (class 0). Mold, Cobweb and Neurospora become **mould** (class 1). Any other class name is dropped and listed, so nothing disappears silently. Photos left with no box are removed (they are not verified clean photos).

In [ ]:
# 7. Map classes and count boxes
import pandas as pd
mould_recs, dropped, rows = [], Counter(), []
no_box = 0
for r in unique:
    boxes = []
    for cls, x, y, w, h in r["boxes"]:
        k = CLASS_MAP.get(cls.strip().lower())
        rows.append((r["src"], cls, OUR_NAMES.get(k, "dropped")))
        if k is None:
            dropped[cls] += 1
        else:
            boxes.append((k, x, y, w, h))
    if boxes:
        mould_recs.append({"path": r["path"], "src": r["src"], "boxes": boxes})
    else:
        no_box += 1
df = pd.DataFrame(rows, columns=["set", "their class", "our class"])
print("boxes per their class:")
display(df.groupby(["their class", "our class"]).size().rename("boxes").reset_index())
print("boxes per our class:")
display(df.groupby("our class").size().rename("boxes"))
print("boxes per set and our class:")
display(pd.crosstab(df["set"], df["our class"]))
print("dropped class names (not mapped):", dict(dropped) or "none")
print(f"photos removed because no mapped box was left: {no_box}")
print(f"MOULD PHOTOS KEPT: {len(mould_recs)}")

## Step 5. Clean ("no mould") dish photos
The clean photos are **d005 only** (the private Kaggle dataset `d005-clean-dishes`). d003 is left out because its images are thin curve strips (about 78×435 px), not whole dishes.

**>>> CHECK THE TWO LINES BELOW <<<** Add the dataset (right panel, *Add Input*) and check `CLEAN_DIR` with the copy icon in the Input panel.

`CLEAN_ID_REGEX` picks the **dish id** out of each file path (one dish = one substrate batch), so that all photos of one dish go to the same side of the split. A few files (for example `FD_0009_Layer 1.jpg`) have no date in the name; they join the dish of the other photos in their folder. The notebook prints photos per dish: check that table. If a dish id cannot be read, it stops instead of guessing.

In [ ]:
# 8. >>> CHECK HERE <<<  clean dish photos (images with no mould = no boxes)
# Check the path with the copy icon in Kaggle's Input panel (next to the dataset name).
CLEAN_DIR = "/kaggle/input/d005-clean-dishes"
# One dish = one substrate batch. Example: FD_0000_180215_FD_Tv_180216.png.jpg -> "180215_FD_Tv"
CLEAN_ID_REGEX = r"(\d{6}_[A-Za-z]+_Tv)"

files = sorted(p for p in Path(CLEAN_DIR).rglob("*") if p.suffix.lower() in IMG_EXT)
assert files, f"no photos found under {CLEAN_DIR}: check that path (copy icon in Kaggle's Input panel)"
bad = [str(p) for p in files if "syldd" in str(p).lower()]
assert not bad, f"syldd photos must not be used here, e.g. {bad[0]}"
by_dish, loose = {}, []
for p in files:
    m = re.search(CLEAN_ID_REGEX, str(p.relative_to(CLEAN_DIR)))
    if m:
        by_dish.setdefault(m.group(1), []).append(p)
    else:
        loose.append(p)
# files without a date in the name (e.g. "FD_0009_Layer 1.jpg") join the one dish found in their folder
for p in loose:
    ids = {k for k, v in by_dish.items() if any(q.parent == p.parent for q in v)}
    if len(ids) != 1:
        raise ValueError(f"cannot read a dish id from {p.name}; fix CLEAN_ID_REGEX (example path: {p.relative_to(CLEAN_DIR)})")
    by_dish[ids.pop()].append(p)
print(f"{len(loose)} file(s) without a date in the name were placed by folder")
print("photos per dish:")
print({k: len(v) for k, v in sorted(by_dish.items())})
print("Note: only about 2 dishes are held out, so the false-alarm count in step 7 is rough.")
assert len(by_dish) >= 5, "fewer than 5 dishes found: the dish id is probably wrong, check CLEAN_ID_REGEX"
dishes = sorted(by_dish)
random.Random(SEED).shuffle(dishes)
n_val = max(1, round(HELD_OUT * len(dishes)))
clean_train, clean_val = [], []
for k, d in enumerate(dishes):
    side = clean_val if k < n_val else clean_train
    side += [{"path": p, "src": "clean", "boxes": []} for p in by_dish[d]]
print(f"clean photos: {len(clean_train)} for training ({len(dishes) - n_val} dishes), {len(clean_val)} held out ({n_val} dishes)")

## EDA of the photos (before training)
**Question: do mould photos and clean photos differ by source or background more than by mould?** The clean photos are all d005 (white dish on brown). If the background alone, or the photo size, tells mould from clean, the model could score well without ever looking at the mould. This cell only looks; nothing is trained and no file is written.

Per **source** (the baibua sets, `mmon` if it is added to `SETS`, and the clean d005 dishes) it gives the same measures as the Harvest readiness EDA (notebook 3.0):

1. **Counts:** photos, boxes per class (Trichoderma / mould), photo sizes.
2. **Quality:** sharpness (variance of a Laplacian on greyscale, long side resized to 1024 px), % blown-out pixels (V > 250), % very dark pixels (V < 15); the 4 worst photos per source.
3. **Colour:** mean V and S (0 to 255), red/blue ratio, hue histogram.
4. **Background:** % of the photo inside boxes (close-up vs far) and the colour of the pixels outside the boxes, by source and by class (a clean photo has no box, so all its pixels count as background).
5. **Source guesser:** a small logistic regression (3x8-bin colour histogram + photo size) that guesses the source, scored with whole groups held out (clean photos by dish; mould photos by photo, as the duplicates are already removed).
6. **Mould vs clean from the background only:** the same kind of model, but it only sees the pixels outside the boxes and the photo size, never the mould itself. AUC 0.5 = it cannot tell; close to 1 = the background gives it away.

Runs in a few minutes on CPU. **Read it:** a high source-guesser score means the sources look very different (so leave-one-source-out will be hard); a high mould-vs-clean AUC means the clean photos can be recognised by background alone, so the false-alarm check in step 7 says little about mould.

In [ ]:
# 8b. EDA of the photos: counts, quality, colour, background, source guesser
import numpy as np, pandas as pd
import cv2
from PIL import Image
LONG_SIDE = 1024      # every photo is resized to this long side before it is measured

def bins8(px):
    # 3 x 8 colour histogram: 8 bins for each of R, G, B; each channel sums to 1 (24 numbers)
    if len(px) == 0:
        return np.full(24, np.nan)
    return np.concatenate([np.bincount(px[:, c] >> 5, minlength=8) / len(px) for c in range(3)])

def measure(path, boxes):
    """boxes = list of (class 0/1, x, y, w, h), as fractions of the photo. Returns numbers for one photo."""
    with Image.open(path) as im:
        W, H = im.size
        exif_rotated = im.getexif().get(274, 1) not in (None, 1)
        rgb = np.asarray(im.convert("RGB"))
    k = LONG_SIDE / max(W, H)
    small = cv2.resize(rgb, (max(1, round(W * k)), max(1, round(H * k))),
                       interpolation=cv2.INTER_AREA if k < 1 else cv2.INTER_LINEAR)
    hh, ww = small.shape[:2]
    grey = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(small, cv2.COLOR_RGB2HSV)          # H 0-179, S 0-255, V 0-255
    Hu, S, V = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    masks = {0: np.zeros((hh, ww), bool), 1: np.zeros((hh, ww), bool)}
    for c, x, y, w, h in boxes:
        x0, x1 = int(np.clip(round((x - w / 2) * ww), 0, ww)), int(np.clip(round((x + w / 2) * ww), 0, ww))
        y0, y1 = int(np.clip(round((y - h / 2) * hh), 0, hh)), int(np.clip(round((y + h / 2) * hh), 0, hh))
        masks[int(c)][y0:y1, x0:x1] = True
    inside = masks[0] | masks[1]
    mean_rgb = small.reshape(-1, 3).mean(axis=0)
    row = {"W": W, "H": H, "exif_rotated": bool(exif_rotated),
           "n_boxes": len(boxes), "n_0": sum(int(b[0]) == 0 for b in boxes), "n_1": sum(int(b[0]) == 1 for b in boxes),
           "cover": 100 * inside.mean(), "cover_0": 100 * masks[0].mean(), "cover_1": 100 * masks[1].mean(),
           "sharp": float(cv2.Laplacian(grey, cv2.CV_64F).var()),
           "blown": 100 * (V > 250).mean(), "dark": 100 * (V < 15).mean(),
           "meanV": float(V.mean()), "meanS": float(S.mean()),
           "rb": float(mean_rgb[0] / max(mean_rgb[2], 1e-6))}
    if inside.any():                                        # colour of the mushrooms / mould themselves
        px = small[inside]
        row["rb_in"] = float(px[:, 0].mean() / max(px[:, 2].mean(), 1e-6))
        row["S_in"] = float(S[inside].mean())
    else:
        row["rb_in"] = row["S_in"] = np.nan
    out_px = small[~inside]                                 # background: everything outside the boxes
    row["out_pct"] = 100 * (~inside).mean()
    if len(out_px) >= 0.01 * hh * ww:
        row["R_out"], row["G_out"], row["B_out"] = (float(v) for v in out_px.mean(axis=0))
        row["V_out"], row["S_out"] = float(V[~inside].mean()), float(S[~inside].mean())
        hist_out = bins8(out_px)
    else:
        row["R_out"] = row["G_out"] = row["B_out"] = row["V_out"] = row["S_out"] = np.nan
        hist_out = np.full(24, np.nan)
    coloured = (S >= 40) & (V >= 40)                        # grey/white/dark pixels have no useful hue
    hue = np.bincount(Hu[coloured] // 5, minlength=36)[:36] / max(coloured.sum(), 1)   # 36 bins of 10 degrees
    return row, bins8(small.reshape(-1, 3)), hist_out, hue

def thumb(path, size=300):
    with Image.open(path) as im:
        im = im.convert("RGB")
        im.thumbnail((size, size))
        return im.copy()

import matplotlib.pyplot as plt

def box_by(ax, df, key, col, order, title, log=False, colours=None):
    # one box per value of `key`, in the given order
    order = [s for s in order if df.loc[df[key] == s, col].notna().any()]
    data = [df.loc[df[key] == s, col].dropna().values for s in order]
    bp = ax.boxplot(data, patch_artist=True, flierprops=dict(markersize=2, alpha=0.4), medianprops=dict(color="black"))
    for patch, s in zip(bp["boxes"], order):
        patch.set_facecolor((colours or {}).get(s, "lightgrey")); patch.set_alpha(0.8)
    ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels(order, rotation=20)
    ax.set_title(title, fontsize=10)
    if log:
        ax.set_yscale("log")

def show_thumbs(groups, title):
    # groups = {name: [(photo path, caption), ...]}; one row per name
    n = max(1, max(len(v) for v in groups.values()))
    fig, axes = plt.subplots(len(groups), n, figsize=(3.2 * n, 2.7 * len(groups)), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for r, (name, items) in enumerate(groups.items()):
        for c, (p, cap) in enumerate(items):
            axes[r][c].imshow(thumb(p)); axes[r][c].set_title(f"{name}\n{cap}", fontsize=7)
    fig.suptitle(title, fontsize=12)
    plt.tight_layout(); plt.show()

from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

def group_cv(X, y, groups, proba=False, max_splits=5):
    """Logistic regression scored out-of-fold. Photos of one group never sit on both sides of a split.
    Returns None when there are too few groups per class to split."""
    y, groups = np.asarray(y), np.asarray(groups)
    n_splits = min([max_splits] + [len(set(groups[y == c])) for c in set(y)])
    if n_splits < 2 or len(set(y)) < 2:
        return None
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
    try:
        out = cross_val_predict(clf, X, y, cv=cv, groups=groups, method="predict_proba" if proba else "predict")
    except ValueError:
        return None
    return out[:, 1] if proba else out

dish_of = {str(p): d for d, ps in by_dish.items() for p in ps}
rows, H_ALL, H_OUT, H_HUE = [], [], [], []
for r in mould_recs + clean_train + clean_val:
    row, ha, ho, hu = measure(r["path"], r["boxes"])          # boxes are (class, x, y, w, h); class 0 = Trichoderma, 1 = mould
    row.update(src="clean d005" if r["src"] == "clean" else r["src"], path=str(r["path"]),
               group=dish_of.get(str(r["path"]), str(r["path"])))
    row["kind"] = ("clean" if row["n_boxes"] == 0 else "Trichoderma only" if row["n_1"] == 0
                   else "mould only" if row["n_0"] == 0 else "both")
    rows.append(row); H_ALL.append(ha); H_OUT.append(ho); H_HUE.append(hu)
PH = pd.DataFrame(rows)
H_ALL, H_OUT, H_HUE = np.array(H_ALL), np.array(H_OUT), np.array(H_HUE)
PH["out_ok"] = np.isfinite(H_OUT).all(axis=1)
PH["is_mould"] = (PH.kind != "clean").astype(int)
SRC = [s for s in SETS if (PH.src == s).any()] + sorted(set(PH.src) - set(SETS) - {"clean d005"}) + ["clean d005"]
SCOL = {s: plt.cm.tab10(i) for i, s in enumerate(SRC)}
print(f"{len(PH)} photos measured, sources: {SRC}")

# ---- 1. counts per class and per source, sizes
print("\n=== 1. Counts per source ===")
rows = {}
for s in SRC:
    d = PH[PH.src == s]
    n0, n1 = int(d.n_0.sum()), int(d.n_1.sum())
    top = ", ".join(f"{w}x{h} ({n})" for (w, h), n in Counter(zip(d.W, d.H)).most_common(3))
    rows[s] = {"photos": len(d), "Trichoderma boxes": n0, "mould boxes": n1,
               "boxes per photo (mean)": round(d.n_boxes.mean(), 1),
               "distinct photo sizes": d[["W", "H"]].drop_duplicates().shape[0],
               "most common sizes WxH (photos)": top, "EXIF-rotated photos": int(d.exif_rotated.sum())}
rows["clean d005"].update({"clean: dishes": len(by_dish), "clean: training photos": len(clean_train), "clean: held-out photos": len(clean_val)})
display(pd.DataFrame(rows).fillna(""))
their = Counter((r["src"], b[0]) for r in unique for b in r["boxes"])
print("their class names per source (boxes, after duplicates removed, before the class mapping):")
display(pd.Series(their).unstack(fill_value=0).T)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
PH.groupby("src").size().reindex(SRC).plot(kind="barh", ax=axes[0], color=[SCOL[s] for s in SRC]); axes[0].set_title("photos per source", fontsize=10)
PH.groupby("src")[["n_0", "n_1"]].sum().reindex(SRC).rename(columns={"n_0": "Trichoderma", "n_1": "mould"}).plot(kind="barh", stacked=True, ax=axes[1], color=["tab:green", "tab:red"])
axes[1].set_title("boxes per class", fontsize=10)
plt.tight_layout(); plt.show()

# ---- 2. photo quality
print("\n=== 2. Photo quality per source ===")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
box_by(axes[0], PH, "src", "sharp", SRC, "sharpness (Laplacian variance, log scale; higher = sharper)", log=True, colours=SCOL)
box_by(axes[1], PH, "src", "blown", SRC, "% blown-out pixels (V > 250)", colours=SCOL)
box_by(axes[2], PH, "src", "dark", SRC, "% very dark pixels (V < 15)", colours=SCOL)
plt.tight_layout(); plt.show()
PH["bad_blur"], PH["bad_blown"], PH["bad_dark"] = 1 - PH.sharp.rank(pct=True), PH.blown.rank(pct=True), PH.dark.rank(pct=True)
PH["bad"] = PH[["bad_blur", "bad_blown", "bad_dark"]].max(axis=1)
why = {"bad_blur": "blurry", "bad_blown": "blown out", "bad_dark": "dark"}
groups = {s: [(r.path, f"{why[r[['bad_blur', 'bad_blown', 'bad_dark']].astype(float).idxmax()]}: sharp {r.sharp:.0f}, blown {r.blown:.1f}%, dark {r.dark:.1f}%")
              for _, r in PH[PH.src == s].nlargest(4, "bad").iterrows()] for s in SRC}
show_thumbs(groups, "4 worst photos per source")

# ---- 3. lighting and colour
print("\n=== 3. Lighting and colour per source ===")
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
box_by(axes[0], PH, "src", "meanV", SRC, "mean brightness V (0-255)", colours=SCOL)
box_by(axes[1], PH, "src", "meanS", SRC, "mean saturation S (0-255)", colours=SCOL)
box_by(axes[2], PH, "src", "rb", SRC, "red/blue ratio, whole photo", colours=SCOL)
for s in SRC:
    axes[3].plot(np.arange(36) * 10 + 5, H_HUE[(PH.src == s).values].mean(axis=0), label=s, color=SCOL[s], lw=2)
axes[3].set_xlabel("hue (degrees: 0 red, 60 yellow, 120 green, 240 blue)"); axes[3].set_title("hue histogram (coloured pixels)", fontsize=10); axes[3].legend(fontsize=7)
plt.tight_layout(); plt.show()
display(PH.groupby("src")[["sharp", "blown", "dark", "meanV", "meanS", "rb"]].median().round(2).loc[SRC])

# ---- 4. background vs class
print("\n=== 4. Background vs class ===")
KINDS = ["Trichoderma only", "mould only", "both", "clean"]
PH["cover_pct"] = PH.cover.clip(lower=0.01)
mp = PH[PH.n_boxes > 0]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
box_by(axes[0], mp, "src", "cover_pct", [s for s in SRC if s != "clean d005"], "% of photo inside boxes (log scale; large = close-up)", log=True, colours=SCOL)
box_by(axes[1], mp, "kind", "cover_pct", KINDS[:3], "same, by class of the photo", log=True, colours={"Trichoderma only": "tab:green", "mould only": "tab:red", "both": "tab:orange"})
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8, 1 + 0.8 * len(SRC)))
img = np.ones((len(SRC), len(KINDS), 3))
for i, s in enumerate(SRC):
    for j, k in enumerate(KINDS):
        d = PH[(PH.src == s) & (PH.kind == k) & PH.R_out.notna()]
        if len(d):
            img[i, j] = d[["R_out", "G_out", "B_out"]].median().values / 255
            ax.text(j, i, f"n={len(d)}\nV {d.V_out.median():.0f}  S {d.S_out.median():.0f}", ha="center", va="center", fontsize=7,
                    color="white" if img[i, j].mean() < 0.5 else "black")
        else:
            ax.text(j, i, "none", ha="center", va="center", fontsize=7)
ax.imshow(img, aspect="auto"); ax.set_xticks(range(len(KINDS))); ax.set_xticklabels(KINDS, fontsize=8)
ax.set_yticks(range(len(SRC))); ax.set_yticklabels(SRC); ax.set_title("median colour of the pixels OUTSIDE the boxes (clean photos: whole photo)", fontsize=10)
plt.tight_layout(); plt.show()
display(PH.groupby(["src", "kind"]).agg(photos=("path", "size"), cover_median=("cover", "median"), V_out=("V_out", "median"), S_out=("S_out", "median")).round(1))

# ---- 5. source guesser
print("\n=== 5. Source guesser (whole groups held out) ===")
y = PH.src.values
size = np.log(PH[["W", "H"]].values)
rows, preds = [], {}
for name, X in {"colour only": H_ALL, "photo size only": size, "colour + size": np.hstack([H_ALL, size])}.items():
    p = group_cv(X, y, PH.group.values)
    if p is None:
        print(name, ": too few groups to split"); continue
    preds[name] = p
    rows.append({"features": name, "accuracy": round(accuracy_score(y, p), 3), "balanced accuracy": round(balanced_accuracy_score(y, p), 3)})
display(pd.DataFrame(rows).set_index("features"))
print(f"always guessing the biggest source: accuracy {PH.src.value_counts(normalize=True).max():.3f}; random guess: balanced accuracy {1 / len(SRC):.3f}")
GUESS = {}
if "colour + size" in preds:
    conf = pd.crosstab(pd.Series(y, name="true source"), pd.Series(preds["colour + size"], name="guessed source")).reindex(index=SRC, columns=SRC, fill_value=0)
    display(conf)
    GUESS = {s: round(100 * conf.loc[s, s] / max(conf.loc[s].sum(), 1)) for s in SRC}

# ---- 6. mould vs clean from the background only (pixels outside the boxes + photo size)
print("\n=== 6. Mould vs clean, background only ===")
ok = PH[PH.out_ok]
Xb = np.hstack([H_OUT[ok.index.values], np.log(ok[["W", "H"]].values)])
rows = []
for name, d, X in [("all mould sources vs clean", ok, Xb)] + \
                  [(f"{s} vs clean", ok[ok.src.isin([s, "clean d005"])], Xb[ok.src.isin([s, "clean d005"]).values]) for s in SRC if s != "clean d005"]:
    yy = d.is_mould.values
    p = group_cv(X, yy, d.group.values, proba=True) if min(np.bincount(yy, minlength=2)) >= 5 else None
    p_size = group_cv(np.log(d[["W", "H"]].values), yy, d.group.values, proba=True) if p is not None else None
    rows.append({"comparison": name, "mould photos": int(yy.sum()), "clean photos": int((1 - yy).sum()),
                 "AUC, background + size": round(roc_auc_score(yy, p), 2) if p is not None else "",
                 "AUC, photo size only": round(roc_auc_score(yy, p_size), 2) if p_size is not None else "",
                 "balanced accuracy, background + size": round(balanced_accuracy_score(yy, p > 0.5), 2) if p is not None else ""})
BGAUC = {r["comparison"].replace(" vs clean", ""): r["AUC, background + size"] for r in rows}
display(pd.DataFrame(rows).set_index("comparison"))

# ---- summary, one row per source
g = PH.groupby("src")
S = pd.DataFrame({"photos": g.size(), "Trichoderma boxes": g.n_0.sum(), "mould boxes": g.n_1.sum(),
                  "main size WxH": {s: f"{Counter(zip(d.W, d.H)).most_common(1)[0][0][0]}x{Counter(zip(d.W, d.H)).most_common(1)[0][0][1]}" for s, d in g},
                  "sharpness": g.sharp.median().round(0), "% blown": g.blown.median().round(2), "% dark": g.dark.median().round(2),
                  "mean V": g.meanV.mean().round(0), "mean S": g.meanS.mean().round(0), "R/B": g.rb.median().round(2),
                  "boxes cover %": mp.groupby("src").cover.median().round(1),
                  "source guessed right %": pd.Series(GUESS), "vs clean, background AUC": pd.Series(BGAUC)}).reindex(SRC)
S[["photos", "Trichoderma boxes", "mould boxes"]] = S[["photos", "Trichoderma boxes", "mould boxes"]].astype(int)
S.index.name = "source"
pd.set_option("display.width", 250); pd.set_option("display.max_columns", None)
print("\n=== SUMMARY (screenshot this) ===")
S

## Step 6. Split, train, score
Mould photos: random 80/20 by photo (duplicates are already gone). Clean photos: split by dish (step 5). The reported precision, recall and mAP50 are on the held-out **mould** photos only; held-out clean photos are used in step 7.

In [ ]:
# 9. Helpers: build a YOLO dataset folder and train / score
def write_split(out, split, items):
    (out / "images" / split).mkdir(parents=True, exist_ok=True)
    (out / "labels" / split).mkdir(parents=True, exist_ok=True)
    for i, r in enumerate(items):
        stem = f"{r['src']}_{i}"
        shutil.copy(r["path"], out / "images" / split / (stem + r["path"].suffix.lower()))
        (out / "labels" / split / (stem + ".txt")).write_text("".join(f"{c} {x} {y} {w} {h}\n" for c, x, y, w, h in r["boxes"]))

def make_dataset(out, train_items, val_items):
    shutil.rmtree(out, ignore_errors=True)
    write_split(out, "train", train_items)
    write_split(out, "val", val_items)
    (out / "data.yaml").write_text(f"path: {out}\ntrain: images/train\nval: images/val\nnames:\n  0: Trichoderma\n  1: mould\n")
    return out / "data.yaml"

def train_and_score(train_items, val_items, tag, epochs=EPOCHS):
    from ultralytics import YOLO
    data = make_dataset(WORK / f"ds_{tag}", train_items, val_items)
    YOLO(MODEL).train(data=str(data), epochs=epochs, imgsz=IMG_SIZE, seed=SEED, project=str(WORK / "runs"),
                      name=tag, exist_ok=True, verbose=False, plots=False)
    best = WORK / "runs" / tag / "weights" / "best.pt"
    model = YOLO(str(best))
    b = model.val(data=str(data), imgsz=IMG_SIZE, verbose=False, plots=False).box
    per_class = {OUR_NAMES[int(c)]: round(float(a), 3) for c, a in zip(b.ap_class_index, b.ap50)}
    scores = {"precision": round(float(b.mp), 3), "recall": round(float(b.mr), 3),
              "mAP50": round(float(b.map50), 3), "mAP50 per class": per_class}
    return model, best, scores

# 10. 80/20 split by photo, then train
shuffled = mould_recs[:]
random.Random(SEED).shuffle(shuffled)
n_val = max(1, round(HELD_OUT * len(shuffled)))
val_mould, train_mould = shuffled[:n_val], shuffled[n_val:]
print(f"mould photos: {len(train_mould)} train, {len(val_mould)} held out")
print("held-out boxes per class:", dict(Counter(OUR_NAMES[b[0]] for r in val_mould for b in r["boxes"])))
model, best_path, scores = train_and_score(train_mould + clean_train, val_mould, "mould")

In [ ]:
# Keep each run's per-epoch history for plotting: one CSV per run in history/, plus the test score, zipped
import datetime, os, re, shutil, yaml
HIST = f"{WORK}/history"
os.makedirs(HIST, exist_ok=True)
TODAY = datetime.date.today()

def save_history(model, run, direction, run_dir, train_set, test_set):
    """Copy run_dir/results.csv (one row per epoch) with the settings in front; returns the file name."""
    df = pd.read_csv(f"{run_dir}/results.csv")
    df.columns = [c.strip() for c in df.columns]
    a = yaml.safe_load(open(f"{run_dir}/args.yaml"))
    front = dict(model=model, run=run, train_set=train_set, test_set=test_set, date=TODAY.isoformat(),
                 epochs=a["epochs"], imgsz=a["imgsz"], batch=a["batch"], base_model=os.path.basename(str(a["model"])))
    for i, (k, v) in enumerate(front.items()):
        df.insert(i, k, v)
    name = "_".join(re.sub(r"[^\w-]+", "", p) for p in (model, run, direction, TODAY.strftime("%Y%m%d"))) + ".csv"
    df.to_csv(f"{HIST}/{name}", index=False)
    return name

def save_scores(model, run, table):
    """The final test score table, next to the histories."""
    t = pd.DataFrame(table)
    for i, (k, v) in enumerate({"model": model, "run": run, "date": TODAY.isoformat()}.items()):
        t.insert(i, k, v)
    t.to_csv(f"{HIST}/{model}_{run}_scores_{TODAY.strftime('%Y%m%d')}.csv", index=False)

train_set = "+".join(sorted({r["src"] for r in train_mould + clean_train}))
print(save_history("mould_detection", "run1", "trainmouldclean_testheldout", str(WORK / "runs" / "mould"),
                   train_set, "heldout20pct_mould"))
save_scores("mould_detection", "run1", [{**{k: v for k, v in scores.items() if k != "mAP50 per class"},
            **{f"AP50 {k}": v for k, v in scores["mAP50 per class"].items()}}])
print(shutil.make_archive(str(WORK / "history"), "zip", HIST))   # Output tab: history.zip

In [ ]:
# 11. Score table (held-out mould photos)
print(json.dumps(scores, indent=2))

## Step 7. False-alarm check
How many held-out **clean** dish photos get any mould box? Lower is better. A confidence of 0.25 is the loose setting, 0.5 the strict one.

In [ ]:
# 12. False alarms on held-out clean photos
false_alarms = {}
if not clean_val:
    print("No held-out clean photos (CLEAN_DIR placeholder not set): false-alarm check skipped.")
else:
    paths = [str(r["path"]) for r in clean_val]
    for conf in (0.25, 0.5):
        hits = 0
        for res in model.predict(paths, conf=conf, imgsz=IMG_SIZE, verbose=False, stream=True):
            hits += len(res.boxes) > 0
        false_alarms[str(conf)] = {"photos with a mould box": hits, "clean photos": len(paths)}
        print(f"confidence {conf}: {hits} of {len(paths)} clean photos get a mould box ({hits / len(paths):.1%})")

## Step 8. Save the weights
`mould_best.pt` and `mould_results.json` appear in the Output tab. Photo folders are deleted so the output stays small.

In [ ]:
# 13. Save weights and results, clean up photo folders
shutil.copy(best_path, WORK / "mould_best.pt")
json.dump({"scores_held_out_mould": scores, "false_alarms_clean": false_alarms,
           "mould_photos": {"train": len(train_mould), "held_out": len(val_mould)},
           "clean_photos": {"train": len(clean_train), "held_out": len(clean_val)},
           "settings": {"model": MODEL, "epochs": EPOCHS, "imgsz": IMG_SIZE, "hash_distance": HASH_DISTANCE, "seed": SEED}},
          open(WORK / "mould_results.json", "w"), indent=2)
for d in ["raw", "ds_mould", "runs"]:
    shutil.rmtree(WORK / d, ignore_errors=True)
print("saved:", sorted(p.name for p in WORK.iterdir()))

## Step 9. Leave one source out (switched off for now)
Today all three sets come from one owner, so leaving one out would test little. Once a **second licensed mould set** is added to `SETS` (and `mould_recs` holds its photos under its own `src` name), remove the `#` marks below. Each source in turn is held out entirely and the model is trained on the others plus the clean training photos. This is the test that shows whether the model works on mould photos from a different place.

In [ ]:
# 14. Leave-one-source-out (uncomment when a second licensed mould set is in)
# loso = []
# for held in sorted({r["src"] for r in mould_recs}):
#     tr = [r for r in mould_recs if r["src"] != held] + clean_train
#     te = [r for r in mould_recs if r["src"] == held]
#     _, _, s = train_and_score(tr, te, f"loso_{held}")
#     loso.append({"left-out source": held, "photos": len(te), **s})
# display(pd.DataFrame(loso))

In [ ]:
# Plot each run's training history: loss and mAP per epoch (matplotlib only; the mycodemo package is not on Kaggle)
import glob
import matplotlib.pyplot as plt
import pandas as pd

for csv in sorted(glob.glob(f"{HIST}/*.csv")):
    if "_scores_" in csv:
        continue
    h = pd.read_csv(csv)
    fig, (axl, axm) = plt.subplots(1, 2, figsize=(12, 4))
    for kind in ["box", "seg", "cls"]:
        for pre, style in [("train", "-"), ("val", "--")]:
            col = f"{pre}/{kind}_loss"
            if col in h and h[col].notna().any():
                axl.plot(h["epoch"], h[col], style, lw=1.6, label=f"{pre} {kind}")
    axl.set(xlabel="epoch", ylabel="loss", title="Loss per epoch (solid train, dashed validation)")
    axl.legend(frameon=False, fontsize=8)
    for col, style in [("metrics/mAP50(B)", "-"), ("metrics/mAP50-95(B)", "--"),
                       ("metrics/mAP50(M)", "-"), ("metrics/mAP50-95(M)", "--")]:
        if col in h and h[col].notna().any():
            axm.plot(h["epoch"], h[col], style, lw=1.8, label=col.replace("metrics/", ""))
    axm.set(xlabel="epoch", ylabel="mAP on the validation photos", ylim=(0, 1), title="Validation mAP per epoch")
    axm.legend(frameon=False, fontsize=8)
    for ax in (axl, axm):
        ax.grid(alpha=0.25)
    fig.suptitle(os.path.basename(csv)[:-4], fontsize=10)
    fig.tight_layout()
    fig.savefig(csv[:-4] + ".png", dpi=130)      # PNG next to its CSV
    plt.show()
print(shutil.make_archive(f"{WORK}/history", "zip", HIST))   # re-zip so history.zip holds the PNGs too